# 01 · Exploração dos Dados

**Autor:** Cristian Matias de Souza · **Projeto:** Social Media Impact

Objetivo: conhecer o dataset de 4.500 estudantes sobre o uso de redes sociais, conferir a qualidade dos dados e entender como o tempo de tela se relaciona com sono, estresse, saúde mental e desempenho acadêmico (GPA).

**Perguntas da análise**
1. Mais horas de uso significam menos sono, mais estresse ou GPA menor?
2. O uso de madrugada (`uso_madrugada`) piora a qualidade do sono?
3. A frequência de comparação social se relaciona com o `indice_saude_mental`?
4. Qual plataforma aparece mais entre os casos de impacto *Negativo*?

> Toda a lógica fica no pacote `src/social_media_impact`. O notebook só **chama** essas funções e mostra os resultados — assim o mesmo código serve para o notebook, para o pipeline e para o dashboard.

## 1 . Carga de Dados


O CSV é lido pela função do pacote, que já confere as colunas e os tipos

In [1]:
import plotly.express as px

from social_media_impact.data import load_raw
from social_media_impact.visualization import (
    colunas_categoricas,
    colunas_numericas,
    contagem,
    grafico_barras,
    grafico_distribuicao,
    para_exibicao,
    resumo_numericas,
    rotulo,
)

In [2]:
# Aqui estamos carregando o dataframe sobre a variavel df (notebook)
df = load_raw()

In [3]:
# Retonando o dataframe e retornando as primeiras linhas
print(df.shape)
df.head(20)

(4500, 16)


,id_estudante,idade,genero,nivel_academico,plataforma_principal,horas_uso_diario,horas_extras_fim_de_semana,tipo_dispositivo,horas_sono,qualidade_sono,uso_madrugada,frequencia_comparacao_social,estresse_percebido,indice_saude_mental,media_academica,impacto_geral
0,STU_2024000,19,Feminino,Graduação,Snapchat,7.5,1.5,Celular,5.9,2,True,Sempre,20.0,73,2.73,Neutro
1,STU_2024001,19,Feminino,Graduação,Instagram,4.6,1.3,Celular,7.9,3,True,Às vezes,17.0,90,3.21,Benéfico
2,STU_2024002,17,Feminino,Ensino médio,TikTok,10.9,1.7,Celular,5.2,1,True,Frequentemente,18.0,70,3.00,Neutro
3,STU_2024003,16,Feminino,Ensino médio,YouTube,6.0,2.3,Tablet,7.6,4,True,Às vezes,15.0,82,3.42,Benéfico
4,STU_2024004,17,Masculino,Ensino médio,LinkedIn,3.3,2.6,Celular,7.6,4,True,Nunca,16.0,84,3.41,Benéfico
5,STU_2024005,16,Masculino,Ensino médio,TikTok,6.2,1.6,Celular,5.9,1,False,Raramente,10.0,66,3.21,Benéfico
6,STU_2024006,20,Masculino,Graduação,TikTok,5.6,1.8,Celular,8.3,3,False,Às vezes,10.0,88,3.72,Benéfico
7,STU_2024007,19,Feminino,Graduação,TikTok,1.2,2.4,Celular,8.1,5,False,Sempre,6.0,95,3.37,Benéfico
8,STU_2024008,18,Feminino,Graduação,YouTube,3.3,1.8,Celular,6.4,2,False,Frequentemente,23.0,88,3.59,Benéfico
9,STU_2024009,18,Feminino,Graduação,Reddit,4.7,1.7,Celular,7.9,2,False,Frequentemente,6.0,96,3.66,Benéfico


## 2. Qualidade dos Dados

In [4]:
# Tipo de cada coluna
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 4500 entries, 0 to 4499
Data columns (total 16 columns):
 #   Column                        Non-Null Count  Dtype   
---  ------                        --------------  -----   
 0   id_estudante                  4500 non-null   str     
 1   idade                         4500 non-null   int64   
 2   genero                        4500 non-null   category
 3   nivel_academico               4500 non-null   category
 4   plataforma_principal          4500 non-null   category
 5   horas_uso_diario              4500 non-null   float64 
 6   horas_extras_fim_de_semana    4500 non-null   float64 
 7   tipo_dispositivo              4500 non-null   category
 8   horas_sono                    4500 non-null   float64 
 9   qualidade_sono                4500 non-null   int64   
 10  uso_madrugada                 4500 non-null   bool    
 11  frequencia_comparacao_social  4500 non-null   category
 12  estresse_percebido            4454 non-null   float64 
 13 

In [5]:
# Valores ausentes: quantidade e % por coluna, só as que têm algum
ausentes = df.isna().sum()
ausentes = ausentes[ausentes > 0].to_frame("qtd")
ausentes["pct"] = (ausentes["qtd"] / len(df) * 100).round(2)
ausentes

,qtd,pct
estresse_percebido,46,1.02
media_academica,85,1.89


In [6]:
# Duplicatas: linhas inteiras e id_estudante repetido
print("Linhas duplicadas:", df.duplicated().sum())
print("id_estudante repetidos:", df["id_estudante"].duplicated().sum())

Linhas duplicadas: 0
id_estudante repetidos: 0


**Achados da qualidade:**
- Tipos corretos; dados já em português.
- Sem linhas nem estudantes duplicados.
- Ausentes só em `estresse_percebido` (46) e `media_academica` (85), menos
  de 2% das linhas.

**Decisão:** não preencher nada. Cada análise que usar estresse ou média
acadêmica descarta os vazios só ali (`dropna` local), sem perder as outras
colunas dessas linhas.

## 3. Visão Geral

### 3.1 Estatísticas das colunas numéricas

In [7]:
# describe() com linhas e estatísticas em português (ver visualization.py)
resumo_numericas(df)

,Qtd,Média,Desvio padrão,Mínimo,1º quartil (25%),Mediana (50%),3º quartil (75%),Máximo
Idade (anos),4500.0,18.70,1.98,15.0,17.0,19.00,20.00,26.0
Uso diário (horas),4500.0,5.30,2.60,0.9,3.4,4.70,6.60,14.0
Horas extras no fim de semana,4500.0,1.80,0.70,0.0,1.3,1.80,2.30,4.5
Sono (horas),4500.0,6.70,1.19,3.0,6.0,6.80,7.50,10.5
Qualidade do sono (1 a 5),4500.0,2.47,1.03,1.0,2.0,3.00,3.00,5.0
Estresse percebido (0 a 40),4454.0,13.51,7.07,0.0,9.0,13.00,18.00,40.0
Índice de saúde mental,4500.0,79.87,12.88,32.0,72.0,82.00,89.00,98.0
Média acadêmica (GPA),4415.0,3.43,0.39,1.9,3.2,3.46,3.72,4.0


**Achados das numéricas:**
- Idade de 15 a 26 anos (mediana 19): público de ensino médio e graduação.
- Uso diário médio de 5,3 h, mas vai de 0,9 h a 14 h; a média fica acima da
  mediana (4,7 h), sinal de cauda longa para a direita (quem usa muito).
- Sono médio de 6,7 h; um quarto dos estudantes dorme 6 h ou menos.
- Saúde mental alta para a maioria (mediana 82), mas com mínimo de 32:
  cauda para a esquerda, de poucos estudantes em situação pior.
- Média acadêmica (GPA) concentrada entre 3,2 e 3,7, com pouca variação.

### 3.2 Colunas Categóricas

In [8]:
categoricas = colunas_categoricas(df)

In [9]:
# Cópia do df só para gráficos e tabelas, com Sim/Não no lugar de True/False
df_exibicao = para_exibicao(df)

In [10]:
# Quantidade e % de cada categoria
for coluna in categoricas:
    print(f"\n{rotulo(coluna)}")
    print(contagem(df_exibicao, coluna))


Gênero
                    qtd   pct
genero                       
Feminino           2234  49.6
Masculino          2051  45.6
Não binário         122   2.7
Prefere não dizer    93   2.1

Nível acadêmico
                  qtd   pct
nivel_academico            
Ensino médio     1477  32.8
Graduação        2777  61.7
Pós-graduação     246   5.5

Plataforma principal
                       qtd   pct
plataforma_principal            
Instagram             1453  32.3
LinkedIn                96   2.1
Reddit                 229   5.1
Snapchat               472  10.5
TikTok                1226  27.2
X (Twitter)            206   4.6
YouTube                818  18.2

Tipo de dispositivo


                   qtd   pct
tipo_dispositivo            
Celular           3775  83.9
Notebook/PC        596  13.2
Tablet             129   2.9

Uso de madrugada
                qtd   pct
uso_madrugada            
Sim            2644  58.8
Não            1856  41.2

Frequência de comparação social
                               qtd   pct
frequencia_comparacao_social            
Nunca                          260   5.8
Raramente                      766  17.0
Às vezes                      1594  35.4
Frequentemente                1248  27.7
Sempre                         632  14.0

Impacto geral


                qtd   pct
impacto_geral            
Negativo        165   3.7
Neutro          654  14.5
Benéfico       3681  81.8


In [11]:
# Gráfico de barras de cada categórica, para ver o peso de cada grupo
for coluna in categoricas:
    grafico_barras(df_exibicao, coluna).show()

**Achados das categóricas:**
 - Gênero equilibrado (49,6% feminino, 45,6% masculino).
 - Maioria na graduação (61,7%); pós-graduação é só 5,5%.
 - Instagram (32,3%) e TikTok (27,2%) somam quase 60% das plataformas.
 - 83,9% usam principalmente o celular.
 - 58,8% usam de madrugada.
 - `impacto_geral` muito desbalanceado: 81,8% Benéfico e só 3,7% Negativo.
   Nas comparações, usar **porcentagens** dentro de cada grupo.

### 3.3 Distribuição das colunas numéricas

In [12]:
# Histograma de cada numérica, com um boxplot em cima (mediana, quartis e extremos)
for coluna in colunas_numericas(df):
    grafico_distribuicao(df, coluna).show()

**Achados das distribuições:**
- _Escreva aqui o que os histogramas mostram (formato, caudas, valores que
  chamam atenção). Pistas: compare o uso diário com a saúde mental: uma tem
  cauda para a direita, a outra para a esquerda._